# Financial Research AI Agent

This notebook builds an AI-powered financial research assistant.

The system can:
- Fetch real-time stock prices
- Analyze stock trends using technical indicators
- Plot stock charts
- Analyze financial news sentiment
- Evaluate portfolio performance
- Provide AI-based financial insights

Technologies used:
- LangChain Agents
- Yahoo Finance API (yfinance)
- Python data analysis (pandas)
- Sentiment analysis (TextBlob)
- Financial visualization (matplotlib)
- Groq LLM

In [ ]:
import os

# Set your Groq API key here
os.environ["GROQ_API_KEY"] = "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"

GROQ_API_KEY = os.environ.get("AI AGENTIC")

print("API Key loaded successfully")

## Step 2: Installing Required Packages

We need to install the necessary LangChain packages and other dependencies for our agent.


In [ ]:
!pip install yfinance pandas matplotlib textblob duckduckgo-search langchain-community langchain-groq gradio ta

In [ ]:
%pip install ta textblob

In [ ]:
import sys
!{sys.executable} -m pip install langchain-groq

In [ ]:
import sys
!{sys.executable} -m pip install langchain langchain-groq langchain-community

In [ ]:
import sys
!{sys.executable} -m pip install langchain==0.1.16 langchain-community==0.0.32

## Step 3: Importing Core Libraries

Let's import the essential LangChain components and other libraries we'll need.


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain.agents import create_agent
import requests
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from textblob import TextBlob
import ta

In [ ]:
import sys
!{sys.executable} -m pip install ddgs

In [ ]:
import sqlite3

In [ ]:
conn = sqlite3.connect("portfolio.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS portfolio(
    symbol TEXT,
    quantity INTEGER
)
""")

conn.commit()

## Step 4: Setting up the Search Tool

We'll use DuckDuckGo search as one of our tools to allow the agent to search for information on the web.


In [ ]:
from duckduckgo_search import DDGS

def search_news(query):
    results = []
    
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=5):
            results.append(r["title"])
    
    return " ".join(results)

## Step 5: Creating a Custom Stock Market Tool

Here we define a custom tool that can fetch weather data for any city using the WeatherStack API. This tool will be available to our agent.


In [ ]:


def get_stock_price(symbol: str) -> str:
    """
    Get the latest stock price for a company.
    """
    if not symbol:
        return "I couldn't identify the exact stock symbol."

    stock = yf.Ticker(symbol)
    
    data = stock.history(period="5d")

    if data.empty:
        return f"Stock data not found for {symbol}. Make sure the ticker is correct."

    price = data["Close"].iloc[-1]

    return f"Latest price of {symbol} is {price:.2f}. (Note: AI must infer the currency based on the exchange it trades on)."

In [ ]:

def analyze_stock_trend(symbol: str) -> str:
    """
    Analyze stock trend using a simple moving average.
    Example: RELIANCE.NS
    """

    stock = yf.Ticker(symbol)
    data = stock.history(period="1mo")

    if data.empty:
        return "No stock data available."

    # Calculate 5-day moving average
    data["MA5"] = data["Close"].rolling(5).mean()

    latest_price = data["Close"].iloc[-1]
    moving_avg = data["MA5"].iloc[-1]

    if latest_price > moving_avg:
        trend = "Uptrend 📈"
    else:
        trend = "Downtrend 📉"

    return f"{symbol} current trend: {trend}. Latest price: ₹{latest_price:.2f}"

In [ ]:

def plot_stock_chart(symbol: str) -> str:
    """
    Plot a stock price chart for the last 3 months.
    Example: TCS.NS
    """

    stock = yf.Ticker(symbol)
    data = stock.history(period="3mo")

    if data.empty:
        return "No stock data available."

    plt.figure(figsize=(10,5))
    plt.plot(data.index, data["Close"])
    plt.title(f"{symbol} Stock Price - Last 3 Months")
    plt.xlabel("Date")
    plt.ylabel("Price (INR)")
    plt.grid(True)

    plt.show()

    return f"Displayed stock chart for {symbol}"

In [ ]:
def analyze_news_sentiment(query: str):

    news_results = search_news(query)

    sentiment_score = TextBlob(news_results).sentiment.polarity

    if sentiment_score > 0:
        sentiment = "Positive 📈"
    elif sentiment_score < 0:
        sentiment = "Negative 📉"
    else:
        sentiment = "Neutral"

    return f"News sentiment for '{query}' appears {sentiment}"

In [ ]:

def analyze_portfolio(symbols: str) -> str:
    """
    Analyze portfolio performance.
    Example input: "TCS.NS, RELIANCE.NS, INFY.NS"
    """

    symbols_list = symbols.split(",")

    results = []

    for sym in symbols_list:
        sym = sym.strip()

        stock = yf.Ticker(sym)
        data = stock.history(period="1mo")

        if data.empty:
            results.append(f"{sym}: No data found")
            continue

        start_price = data["Close"].iloc[0]
        end_price = data["Close"].iloc[-1]

        change = ((end_price - start_price) / start_price) * 100

        results.append(f"{sym}: {change:.2f}% change in last month")

    return "\n".join(results)

In [ ]:
def technical_analysis(symbol):
    # Use Ticker().history() for consistency and a 1D shape
    stock = yf.Ticker(symbol)
    data = stock.history(period="3mo")

    if data.empty:
        return f"No technical data found for {symbol}."

    # Force the 'Close' column to be a flat 1-dimensional line of numbers
    close_prices = data['Close'].squeeze()

    data['MA20'] = close_prices.rolling(window=20).mean()
    data['MA50'] = close_prices.rolling(window=50).mean()

    # Pass the perfectly flat 1D data to the math library
    rsi = ta.momentum.RSIIndicator(close_prices)
    data['RSI'] = rsi.rsi()

    latest = data.iloc[-1]

    return f"""
Technical Analysis for {symbol}

Latest Price: {latest['Close']:.2f}

MA20: {latest['MA20']:.2f}
MA50: {latest['MA50']:.2f}

RSI: {latest['RSI']:.2f}

(Note: RSI > 70 → Overbought, RSI < 30 → Oversold)
"""

In [ ]:
def compare_stocks(symbol1, symbol2):

    s1 = yf.download(symbol1, period="1mo")
    s2 = yf.download(symbol2, period="1mo")

    p1 = s1["Close"].iloc[-1]
    p2 = s2["Close"].iloc[-1]

    change1 = ((p1 - s1["Close"].iloc[0]) / s1["Close"].iloc[0]) * 100
    change2 = ((p2 - s2["Close"].iloc[0]) / s2["Close"].iloc[0]) * 100

    return f"""
Stock Comparison

{symbol1}
Price: ₹{p1:.2f}
1 Month Change: {change1:.2f}%

{symbol2}
Price: ₹{p2:.2f}
1 Month Change: {change2:.2f}%

Better performer this month: {symbol1 if change1 > change2 else symbol2}
"""

In [ ]:
def moving_average_signal(symbol):

    data = yf.download(symbol, period="1y")

    data["MA50"] = data["Close"].rolling(window=50).mean()
    data["MA200"] = data["Close"].rolling(window=200).mean()

    ma50 = data["MA50"].iloc[-1]
    ma200 = data["MA200"].iloc[-1]

    if ma50 > ma200:
        signal = "BUY signal 📈 (Golden Cross)"
    elif ma50 < ma200:
        signal = "SELL signal 📉 (Death Cross)"
    else:
        signal = "HOLD"

    return f"""
Technical Signal for {symbol}

50 Day Moving Average: {ma50:.2f}
200 Day Moving Average: {ma200:.2f}

Trading Signal: {signal}
"""

In [ ]:
def research_report(symbol):

    price = get_stock_price(symbol)
    trend = analyze_stock_trend(symbol)
    tech = technical_analysis(symbol)
    signal = moving_average_signal(symbol)
    sentiment = analyze_news_sentiment(symbol)

    return f"""
📊 STOCK RESEARCH REPORT

Symbol: {symbol}

--------------------------------

PRICE
{price}

--------------------------------

TREND ANALYSIS
{trend}

--------------------------------

TECHNICAL INDICATORS
{tech}

--------------------------------

TRADING SIGNAL
{signal}

--------------------------------

NEWS SENTIMENT
{sentiment}

--------------------------------

AI Summary:
Based on current technical indicators and sentiment, review the trend and signal before making investment decisions.
"""

In [ ]:
def add_to_portfolio(symbol, quantity):

    conn = sqlite3.connect("portfolio.db")
    cursor = conn.cursor()

    cursor.execute(
        "INSERT INTO portfolio VALUES (?,?)",
        (symbol, quantity)
    )

    conn.commit()
    conn.close()

    return f"{quantity} shares of {symbol} added to portfolio."

In [ ]:
def view_portfolio():

    conn = sqlite3.connect("portfolio.db")
    cursor = conn.cursor()

    cursor.execute("SELECT * FROM portfolio")

    rows = cursor.fetchall()

    conn.close()

    if not rows:
        return "Portfolio is empty."

    result = "📊 Your Portfolio\n\n"

    for symbol, quantity in rows:
        result += f"{symbol} : {quantity} shares\n"

    return result

In [ ]:
def portfolio_value():

    conn = sqlite3.connect("portfolio.db")
    cursor = conn.cursor()

    cursor.execute("SELECT * FROM portfolio")

    rows = cursor.fetchall()

    conn.close()

    if not rows:
        return "Portfolio is empty."

    total = 0
    report = "📊 Portfolio Value\n\n"

    for symbol, quantity in rows:

        data = yf.download(symbol, period="1d")
        price = data["Close"].iloc[-1]

        value = price * quantity
        total += value

        report += f"{symbol} : {quantity} shares | ₹{value:.2f}\n"

    report += f"\nTotal Portfolio Value: ₹{total:.2f}"

    return report

## Step 6: Initializing the Language Model

We create an instance of the ChatOpenAI model that will power our agent's reasoning capabilities.


In [ ]:
# llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
llm = ChatGroq(
    model="mixtral-8x7b-32768",
    temperature=0,
    # Enable tool calling for this model
    model_kwargs={"tool_choice": "auto"},
    api_key=GROQ_API_KEY
)

In [ ]:
from langchain_core.messages import HumanMessage

def detect_symbol(query):
    """
    Uses the AI model to dynamically find the Yahoo Finance ticker for ANY asset.
    """
    prompt = f"""
    Analyze this user query: "{query}"
    Identify the asset (company, index, or cryptocurrency) mentioned and provide its exact Yahoo Finance ticker symbol. 
    - If it's an Indian company, append the NSE suffix (.NS) (e.g., RELIANCE.NS, ZOMATO.NS). 
    - If it's a US company, provide the standard US ticker (e.g., AAPL, TSLA).
    - If it's a cryptocurrency, provide the USD ticker (e.g., BTC-USD, ETH-USD).
    - If the company is PRIVATE and not publicly traded on the stock market (e.g., Physics Wallah, Zepto), reply with the word NONE.
    
    Reply ONLY with the ticker symbol. Do not add any other words, punctuation, or explanations.
    If no valid asset is mentioned, reply with the word NONE.
    """
    
    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        symbol = response.content.strip(" \n\"'.`").upper()
        
        print(f"DEBUG - AI Detected Symbol: >>>{symbol}<<<")
        
        if "NONE" in symbol or symbol == "":
            return None
            
        return symbol
        
    except Exception as e:
        print(f"Error detecting symbol: {e}")
        return None

In [ ]:
def detect_quantity(query):

    words = query.split()

    for word in words:
        if word.isdigit():
            return int(word)

    return None

## Step 9: Creating the Agent

Now we create our agent by combining the language model, our tools (search and weather), and the ReAct prompt template.


In [ ]:
%pip install langgraph

In [ ]:
from langchain_core.tools import tool

# 1. We wrap your existing functions in @tool so the LLM can "see" and use them
@tool
def get_stock_price_tool(symbol: str) -> str:
    """Gets the latest live stock price for a company. Input MUST be a valid Yahoo Finance ticker symbol (e.g., RELIANCE.NS, AAPL)."""
    return get_stock_price(symbol)

@tool
def analyze_stock_trend_tool(symbol: str) -> str:
    """Analyzes if a stock is in an uptrend or downtrend using moving averages. Input MUST be a valid ticker symbol."""
    return analyze_stock_trend(symbol)

@tool
def analyze_news_tool(query: str) -> str:
    """Searches the web for the latest news on a company and returns the sentiment (Positive/Negative/Neutral). Input can be a company name."""
    return analyze_news_sentiment(query)

@tool
def technical_analysis_tool(symbol: str) -> str:
    """Provides technical indicators like RSI, MA20, and MA50 for a stock to determine if it is overbought or oversold. Input MUST be a valid ticker."""
    return technical_analysis(symbol)

# 2. Bundle them into a list for the LangGraph agent
financial_tools = [get_stock_price_tool, analyze_stock_trend_tool, analyze_news_tool, technical_analysis_tool]

In [ ]:
from langgraph.prebuilt import create_react_agent

# 1. Define the AI's Persona 
system_prompt = """You are an elite, professional financial AI assistant. 
You have access to real-time financial tools. 
Always use your tools to fetch live data before answering. 
If a user asks a complex question, break it down step-by-step.
If a user makes spelling mistakes or typos in their prompt, intelligently infer their true intent and auto-correct stock names to the proper Yahoo Finance tickers (append .NS for Indian stocks).
If a user asks a complex question, break it down step-by-step.
If you need a ticker symbol for a tool, use your internal knowledge to guess the correct Yahoo Finance ticker (append .NS for Indian stocks)."""

# 2. Build the Agent (Notice we are using 'prompt=' instead of 'state_modifier=')
agent_executor = create_react_agent(llm, financial_tools, prompt=system_prompt)

# 3. Our wrapper function for Gradio
def financial_agent(query: str):
    try:
        # LangGraph uses a 'messages' list instead of a generic 'input'
        response = agent_executor.invoke({"messages": [("user", query)]})
        
        # The AI's final answer is always the very last message in the sequence
        return response["messages"][-1].content
    except Exception as e:
        return f"An error occurred while analyzing: {e}"

In [ ]:
from langchain_core.tools import tool

# 1. We wrap your existing functions in @tool so the LLM can "see" and use them
@tool
def get_stock_price_tool(symbol: str) -> str:
    """Gets the latest live stock price for a company. Input MUST be a valid Yahoo Finance ticker symbol (e.g., RELIANCE.NS, AAPL)."""
    return get_stock_price(symbol)

@tool
def analyze_stock_trend_tool(symbol: str) -> str:
    """Analyzes if a stock is in an uptrend or downtrend using moving averages. Input MUST be a valid ticker symbol."""
    return analyze_stock_trend(symbol)

@tool
def analyze_news_tool(query: str) -> str:
    """Searches the web for the latest news on a company and returns the sentiment (Positive/Negative/Neutral). Input can be a company name."""
    return analyze_news_sentiment(query)

@tool
def technical_analysis_tool(symbol: str) -> str:
    """Provides technical indicators like RSI, MA20, and MA50 for a stock to determine if it is overbought or oversold. Input MUST be a valid ticker."""
    return technical_analysis(symbol)

# 2. Bundle them into a list for the agent to access
financial_tools = [get_stock_price_tool, analyze_stock_trend_tool, analyze_news_tool, technical_analysis_tool]

## Step 10: Testing the Agent

Let's test our agent with a complex query that requires both searching for information and getting weather data. The agent will need to:
1. Find the capital of Madhya Pradesh
2. Get the current weather for that city


In [ ]:
response = financial_agent("price of RELIANCE.NS")
print(response)

In [ ]:
import yfinance as yf
data = yf.Ticker("ZOMATO.NS").history(period="5d")
print(data)

# Final Step: Deployment

We'll use gradio to deploy our AI Agents. To deploy we'll follow the following steps.

1. Add all our logic for AI Agent into a single function
2. Create an interface using gradio's classes
3. Launch the app

In [ ]:
%pip install gradio

In [ ]:
import gradio as gr

In [ ]:
def get_response(query):
    return financial_agent(query)

In [ ]:
iface = gr.Interface(
    fn=get_response,
    inputs=gr.Textbox(
        label="Ask a question to your AI Agent",
        placeholder="Ask about stock price, trend, or portfolio...",
        lines=2
    ),
    outputs=gr.Textbox(label="Response", lines=10),
    title="AI Agent with Web Access",
    description="This AI Agent has access to the internet. You can ask anything and it will search the web to get you your answer.",
    examples=[
        ["Differentiate between VectorDB and Vector Store"],
        ["What is RAG model?"],
        ["What is the current market cap of NVIDIA"]
    ]
)

In [ ]:
iface.launch(share=True)

## Summary

This notebook demonstrated how to:
- Set up LangChain agents with custom tools
- Use the ReAct framework for reasoning and acting
- Combine multiple tools (search and weather API) in a single agent
- Execute complex multi-step queries that require both information retrieval and data processing
- Deploy an AI Agent using gradio

The agent successfully found that Bhopal is the capital of Madhya Pradesh and retrieved its current weather conditions by using both the search tool and the weather API tool in sequence.
